# YOLO + ArUco + 경로 예측 + 절대좌표 전송 (v6)

## v5 대비 변경점
- 좌표 추출 기준 변경: 박스 중심 → 박스 밑변 중심 (발·바퀴 위치)
- 픽셀 → 절대좌표 변환 함수 자리 마련 (목업)
- 5프레임마다 딥러닝 모델로 JSON 전송하는 목업 코드 추가
- 위치 추적 기준도 밑변 중심으로 통일 (예측 연속성 확보)

## 담당 범위
- 본 코드: 객체 검출·추적·예측·절대좌표 수집·JSON 출력
- 별도 조원: 픽셀 → 절대좌표 변환 로직 (Homography)
- 딥러닝 팀: 좌표 수신·위험도 판정·MQTT 발송

In [16]:
import cv2
import os
import time
import math
import json
import numpy as np
from collections import deque
from ultralytics import YOLO
from dotenv import load_dotenv

load_dotenv()
best_model_path = os.getenv("best_model_path")

In [17]:
# ========================================
# 경로 예측 파라미터 (시간 기반)
# ========================================

HISTORY_SECONDS = 0.5
PREDICT_SECONDS = 2.0
MIN_HISTORY_SECONDS = 0.3

COLLISION_RADIUS = 250

YOLO_IMGSZ = 480
WINDOW_W = 1600
WINDOW_H = 900

WINDOW_NAME = "Pose + Forklift + ArUco + Prediction"

# ========================================
# 딥러닝 전송 설정
# ========================================

CAMERA_ID = 1              # 이 노트북이 담당하는 카메라 번호 (1, 2, 3 중)
SEND_EVERY_N_FRAMES = 5    # 몇 프레임마다 딥러닝 모델에 전송할지

print(f"설정 완료: {PREDICT_SECONDS}초 후 예측 / {SEND_EVERY_N_FRAMES}프레임마다 전송")

설정 완료: 2.0초 후 예측, 0.5초 궤적 관찰


In [18]:
# ========================================
# 모델 로드
# ========================================

pose_model = YOLO("yolo11n-pose.pt")
forklift_model = YOLO(best_model_path)

aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_5X5_50)
aruco_params = cv2.aruco.DetectorParameters()
aruco_detector = cv2.aruco.ArucoDetector(aruco_dict, aruco_params)

worker_map = {
    0: "Worker1",
    22: "Worker2",
    24: "Worker3",
    27: "Worker4",
    38: "Worker5",
}

track_to_worker = {}
print("모델 로드 완료")

모델 로드 완료


In [19]:
# ========================================
# 좌표 추출 · 변환 함수
# ========================================

def get_bottom_center(x1, y1, x2, y2):
    """박스 밑변 중심 좌표 반환 (발·바퀴 위치).
    - 작업자: 양발 중심
    - 지게차: 밑바닥 가로 중심
    """
    cx = (x1 + x2) // 2
    cy = y2   # 박스 아래쪽 y값
    return (cx, cy)


def pixel_to_absolute(pixel_x, pixel_y, camera_id=CAMERA_ID):
    """
    ===========================================================
    TODO: 별도 조원이 구현할 픽셀 → 절대좌표 변환 함수.
    - 바닥 ArUco 4개 Homography 행렬 사용
    - 단위: meter
    - 카메라별 Homography가 달라 camera_id로 구분
    
    예상 시그니처:
        return (abs_x_m, abs_y_m)
    ===========================================================
    
    현재는 목업: 픽셀값을 100으로 나눈 값을 임시 미터값으로 반환.
    실제 구현 전까지 구조 검증용.
    """
    # ---------- 실제 구현으로 교체 예정 ----------
    abs_x = round(pixel_x / 100.0, 2)
    abs_y = round(pixel_y / 100.0, 2)
    return (abs_x, abs_y)
    # --------------------------------------------


# ========================================
# 예측 관련 함수
# ========================================

def update_history(history_dict, obj_id, cx, cy, max_len):
    if obj_id not in history_dict:
        history_dict[obj_id] = deque(maxlen=max_len)
    elif history_dict[obj_id].maxlen != max_len:
        old_data = list(history_dict[obj_id])
        history_dict[obj_id] = deque(old_data, maxlen=max_len)
    history_dict[obj_id].append((cx, cy))


def predict_future_position(history, predict_frames, min_history):
    if len(history) < min_history:
        return None

    old_x, old_y = history[0]
    cur_x, cur_y = history[-1]
    n_frames = len(history) - 1

    if n_frames == 0:
        return None

    vx = (cur_x - old_x) / n_frames
    vy = (cur_y - old_y) / n_frames

    future_x = int(cur_x + vx * predict_frames)
    future_y = int(cur_y + vy * predict_frames)

    return (future_x, future_y)


def is_collision_risk(pos1, pos2, radius=COLLISION_RADIUS):
    if pos1 is None or pos2 is None:
        return False
    distance = np.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)
    return distance < radius


def draw_prediction(frame, current_pos, future_pos, color, label=""):
    if future_pos is None:
        return

    cx, cy = current_pos
    fx, fy = future_pos

    num_dots = 10
    for i in range(num_dots):
        t1 = i / num_dots
        t2 = (i + 0.5) / num_dots
        p1 = (int(cx + (fx - cx) * t1), int(cy + (fy - cy) * t1))
        p2 = (int(cx + (fx - cx) * t2), int(cy + (fy - cy) * t2))
        cv2.line(frame, p1, p2, color, 2)

    cv2.circle(frame, (fx, fy), 15, color, 2)

    if label:
        cv2.putText(frame, label, (fx + 18, fy),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)


# ========================================
# 딥러닝 모델 전송 (목업)
# ========================================

def send_to_deeplearning(payload):
    """
    ===========================================================
    TODO: 딥러닝 모델로 JSON payload 전송.
    실제 구현 시 선택:
      · HTTP POST (requests 사용)
      · WebSocket
      · ZeroMQ / socket
      · 파일 queue
    
    팀원과 프로토콜 확정 후 구현.
    ===========================================================
    
    현재는 목업: 콘솔에 출력만.
    """
    # ---------- 실제 전송 코드 자리 ----------
    # requests.post("http://딥러닝서버/api/coords", json=payload)
    # 또는 websocket.send(json.dumps(payload))
    # -----------------------------------------
    
    print(f"[SEND] {json.dumps(payload, ensure_ascii=False)}")


print("함수 정의 완료")

함수 정의 완료


In [20]:
# ========================================
# 메인 루프
# ========================================

worker_history = {}
forklift_history = {}

cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
cv2.resizeWindow(WINDOW_NAME, WINDOW_W, WINDOW_H)

cap = cv2.VideoCapture(0)

# FPS 측정용
fps_counter = 0
fps_start_time = time.time()
current_fps = 30.0

# 프레임 카운터 (5프레임마다 전송)
frame_count = 0

# 색상 상수
NORMAL_COLOR = (0, 255, 0)
DANGER_COLOR = (0, 0, 255)

print("루프 시작 - ESC 누르면 종료")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # FPS 측정
    fps_counter += 1
    elapsed = time.time() - fps_start_time
    if elapsed >= 1.0:
        current_fps = fps_counter / elapsed
        fps_counter = 0
        fps_start_time = time.time()

    # FPS 기반 동적 파라미터
    history_len = max(int(HISTORY_SECONDS * current_fps), 5)
    predict_frames = int(PREDICT_SECONDS * current_fps)
    min_history = max(int(MIN_HISTORY_SECONDS * current_fps), 3)

    # YOLO 추론
    pose_results = pose_model.track(
        frame, conf=0.25, persist=True, verbose=False, classes=[0],
        imgsz=YOLO_IMGSZ
    )
    forklift_results = forklift_model(
        frame, conf=0.5, verbose=False,
        imgsz=YOLO_IMGSZ
    )

    # ============================================
    # 작업자 박스 + 궤적 (밑변 중심 기준)
    # ============================================
    person_boxes = []
    if (pose_results[0].boxes is not None
            and pose_results[0].boxes.id is not None):
        xyxy = pose_results[0].boxes.xyxy.cpu().numpy()
        ids_t = pose_results[0].boxes.id.cpu().numpy().astype(int)
        for box, tid in zip(xyxy, ids_t):
            x1, y1, x2, y2 = box.astype(int)
            person_boxes.append((tid, x1, y1, x2, y2))

            # ★ 밑변 중심 (양발 위치)
            bx, by = get_bottom_center(x1, y1, x2, y2)
            update_history(worker_history, tid, bx, by, history_len)

    # ============================================
    # 지게차 박스 + 궤적 (밑변 중심 기준)
    # ============================================
    forklift_boxes = []
    for i, box in enumerate(forklift_results[0].boxes):
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls_id = int(box.cls[0]) if hasattr(box, 'cls') else 0
        forklift_boxes.append((i, x1, y1, x2, y2, cls_id))

        # ★ 밑변 중심 (가로 중심, 바닥 접지면)
        bx, by = get_bottom_center(x1, y1, x2, y2)
        update_history(forklift_history, i, bx, by, history_len)

    # ============================================
    # ArUco 검출 + 매칭
    # ============================================
    corners, ids, _ = aruco_detector.detectMarkers(frame)
    if ids is not None:
        for marker_corners, marker_id in zip(corners, ids.flatten()):
            marker_id = int(marker_id)
            if marker_id not in worker_map:
                continue
            pts = marker_corners[0]
            cx = int(pts[:, 0].mean())
            cy = int(pts[:, 1].mean())
            for tid, x1, y1, x2, y2 in person_boxes:
                if x1 <= cx <= x2 and y1 <= cy <= y2:
                    track_to_worker[tid] = worker_map[marker_id]
                    break

    # ============================================
    # 시각화
    # ============================================
    annotated_frame = pose_results[0].plot()

    # 지게차 박스 + 밑변 중심 점 표시
    for i, x1, y1, x2, y2, cls_id in forklift_boxes:
        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
        label = "forklift" if cls_id == 0 else f"box_{cls_id}"
        cv2.putText(annotated_frame, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
        # 밑변 중심 점 (디버깅·확인용)
        bx, by = get_bottom_center(x1, y1, x2, y2)
        cv2.circle(annotated_frame, (bx, by), 4, (0, 255, 255), -1)

    # 작업자 이름 + 밑변 중심 점
    for tid, x1, y1, x2, y2 in person_boxes:
        name = track_to_worker.get(tid)
        if name is None:
            display_text = "Unknown"
            color = (128, 128, 128)
        else:
            display_text = name
            color = (0, 255, 255)
        cv2.putText(annotated_frame, display_text, (x1, y1 - 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
        # 밑변 중심 점
        bx, by = get_bottom_center(x1, y1, x2, y2)
        cv2.circle(annotated_frame, (bx, by), 4, (0, 255, 255), -1)

    if ids is not None:
        cv2.aruco.drawDetectedMarkers(annotated_frame, corners, ids)

    # ============================================
    # 경로 예측 + 충돌 판정 + 시각화
    # ============================================
    worker_predictions = {}
    forklift_predictions = {}
    pred_label = f"+{PREDICT_SECONDS:.1f}s"

    for tid, x1, y1, x2, y2 in person_boxes:
        if tid not in worker_history:
            continue
        future = predict_future_position(
            worker_history[tid], predict_frames, min_history
        )
        worker_predictions[tid] = future

    for i, x1, y1, x2, y2, cls_id in forklift_boxes:
        if i not in forklift_history:
            continue
        future = predict_future_position(
            forklift_history[i], predict_frames, min_history
        )
        forklift_predictions[i] = future

    # 충돌 판정
    collision_worker_ids = set()
    collision_forklift_ids = set()
    for tid, w_future in worker_predictions.items():
        for idx, f_future in forklift_predictions.items():
            if is_collision_risk(w_future, f_future):
                collision_worker_ids.add(tid)
                collision_forklift_ids.add(idx)

    # 작업자 예측 원
    for tid, x1, y1, x2, y2 in person_boxes:
        if tid not in worker_history:
            continue
        current = worker_history[tid][-1]
        future = worker_predictions.get(tid)
        color = DANGER_COLOR if tid in collision_worker_ids else NORMAL_COLOR
        draw_prediction(annotated_frame, current, future,
                        color=color, label=pred_label)

    # 지게차 예측 원
    for i, x1, y1, x2, y2, cls_id in forklift_boxes:
        if i not in forklift_history:
            continue
        current = forklift_history[i][-1]
        future = forklift_predictions.get(i)
        color = DANGER_COLOR if i in collision_forklift_ids else NORMAL_COLOR
        draw_prediction(annotated_frame, current, future,
                        color=color, label=pred_label)

    # ============================================
    # 5프레임마다 딥러닝 모델로 전송 (목업)
    # ============================================
    if frame_count % SEND_EVERY_N_FRAMES == 0:
        objects_payload = []

        # 작업자
        for tid, x1, y1, x2, y2 in person_boxes:
            bx, by = get_bottom_center(x1, y1, x2, y2)
            abs_x, abs_y = pixel_to_absolute(bx, by)
            name = track_to_worker.get(tid, f"Unknown_{tid}")
            objects_payload.append({
                "type": "worker",
                "id": name,
                "center_abs": [abs_x, abs_y],
            })

        # 지게차·박스
        for i, x1, y1, x2, y2, cls_id in forklift_boxes:
            bx, by = get_bottom_center(x1, y1, x2, y2)
            abs_x, abs_y = pixel_to_absolute(bx, by)
            type_name = {0: "forklift", 1: "box_1", 2: "box_2"}.get(cls_id, "unknown")
            objects_payload.append({
                "type": type_name,
                "id": f"{type_name}_{i:02d}",
                "center_abs": [abs_x, abs_y],
            })

        # 페이로드 최종 조립
        payload = {
            "timestamp": round(time.time(), 3),
            "frame_id": frame_count,
            "camera_id": CAMERA_ID,
            "objects": objects_payload,
        }

        send_to_deeplearning(payload)

    cv2.imshow(WINDOW_NAME, annotated_frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

print(f"종료. 최종 FPS: {current_fps:.1f}")

루프 시작 - ESC 누르면 종료
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
종료. 최종 FPS: 4.1
